In [1]:
!pip install huggingface
!pip install huggingface_hub
!pip install transformers

In [2]:
from datasets import Dataset, DatasetDict
import pandas as pd
df1 = pd.read_csv("Rajasthan_metadata.csv")
df2 = pd.read_csv("Rest_metadata.csv")
df1 = df1[['Name','Caste','Clustered_Caste']]
df2 = df2[['Name']]
df1['Caste'] = df1['Caste'].fillna('Unknown')
df1['Clustered_Caste'] = df1['Clustered_Caste'].fillna('Unknown')

name1 = []
caste1 = []
cluster1 = []
name2 = []
for i in range(len(df1)):
  name1.append(df1['Name'].iloc[i])
  caste1.append(df1['Caste'].iloc[i])
  cluster1.append(df1['Clustered_Caste'].iloc[i])
for i in range(len(df2)):
  name2.append(df2['Name'].iloc[i])
dict1 = {'Name':name1,'Caste':caste1}

dict2 = {'Name':name2}
train_dataset = Dataset.from_dict(dict1)
test_dataset = Dataset.from_dict(dict2)
dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

from sklearn.preprocessing import LabelEncoder
le_caste = LabelEncoder()
dataset["train"] = dataset["train"].remove_columns("Caste").add_column(
    "label_caste", le_caste.fit_transform(dataset["train"]["Caste"])
)
print(dataset)
num_labels = len(le_caste.classes_)
print(num_labels)

DatasetDict({
    train: Dataset({
        features: ['Name', 'label_caste'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['Name'],
        num_rows: 666
    })
})
875


In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Name', 'label_caste'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['Name'],
        num_rows: 666
    })
})


In [4]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
def tokenize(examples):
    return tokenizer(
        examples["Name"],
        padding="max_length",
        truncation=True
    )

dataset = dataset.map(tokenize, batched=True)
model = AutoModelForSequenceClassification.from_pretrained(
    "ai4bharat/indic-bert",
    num_labels=num_labels
)
import numpy as np


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/666 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.decoder.bias         | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
classifier.weight                | MISSING    | 
classifier.bias                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Name', 'label_caste', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['Name', 'input_ids', 'attention_mask'],
        num_rows: 666
    })
})


In [8]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00


model.safetensors:   0%|          | 0.00/135M [00:00<?, ?B/s]

In [9]:
import evaluate
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)
from transformers import TrainingArguments, DataCollatorWithPadding # Re-add DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="yelp_review_classifier",
    eval_strategy="epoch",
    push_to_hub=False,
    warmup_steps=0.13,
    learning_rate = 5e-5,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=3e-8,
    weight_decay=0.001,
    lr_scheduler_type = "linear",
    num_train_epochs=50
)
from transformers import Trainer

split = dataset["train"].train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]
print(train_dataset)
train_dataset = train_dataset.rename_column("label_caste", "labels")
eval_dataset = eval_dataset.rename_column("label_caste", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

trainer.train()

pred_results = trainer.predict(dataset["test"])
pred_labels = np.argmax(pred_results.predictions, axis=-1)
pred_categories = [le_caste.classes_[i] for i in pred_labels]
dict2["Caste"] = pred_categories
print(dict2)

Dataset({
    features: ['Name', 'label_caste', 'input_ids', 'attention_mask'],
    num_rows: 4500
})


Epoch,Training Loss,Validation Loss,Accuracy
1,6.670458,5.819726,0.088000
2,5.524766,5.203301,0.088000
3,5.045006,5.049776,0.162000
4,4.908626,5.091209,0.162000
5,4.903917,4.936010,0.206000
6,4.704751,4.881979,0.218000
7,4.641615,4.844753,0.218000
8,4.501277,4.839772,0.214000
9,4.404791,4.884697,0.222000
10,4.405889,4.837823,0.202000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'Name': ['ABHAY', 'PANCHAL MANOJ ', 'GONSALVES SUNNY', 'AMRIT', 'BAISOYA RISHABH', 'GIRI HEMANT', 'KADIAN YOGESH', 'SINGH HARBHEJ', 'SINGH JOGA', 'SHARMA SACHIN', 'BAJAJ RITIK', 'SINGH BARJINDER', 'KUMAR ANIL', 'PATEL PRITESH MUKESHBHAI', 'JAIN HARSHIT BABULAL', 'SINGH VIKRAMJEET', 'SANGWAN KAPIL', 'SINGH SATINDERJEET', 'SANDHU HARWINDER SINGH', 'SINGH RANDEEP', 'KOLANJA MOHAMMAD RAFEEQ', 'PURUSHAN SUDHEESH KUMAR', 'HAIDER SYED MUHAMMAD ARSHIYAN', 'NANAVATI ADITYA', 'SINGH GURJINDER', 'GUPTA VIKASH', 'SANDHU SHAMINDER SINGH', 'RAIJADA KAVALJITSINH MAHENDRASINH', 'SINGH HARSAHIB', 'ANGOM SANATOMBA SINGH', 'PUKHRAMBAM SAMANANDA SINGH', 'KSHETRIMAYUM BROJEN SINGH', 'TAKHELAMBAM RAGHU', 'THOKCHOM GANDHI SINGH', 'SULTAN AYAZ MOHAMMED', 'ARMAR MOHAMMAD SHAFI', 'SINGH KARANVIR', 'SINGH AVTAR', 'SINGH HARMEET', 'RANPARIYA JAYSUKH', 'SINGH JANG BAHADUR', 'SINGH GURMIT', 'BHANSALI MIHIR RASHMI', 'SINGH GURPREET', 'SINGH MEHENGA', 'LAMKANG STARSON', 'MOIRANGTHEM TAMBA', 'KHURAIJAM MAIPAK', 'MUNS

In [13]:

def tokenize(examples):
    return tokenizer(
        examples["Name"],
        padding="max_length",   # or padding=True for dynamic padding
        truncation=True
    )

test_dataset = dataset["test"].map(tokenize, batched=True)
pred_results = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_results.predictions, axis=-1)
pred_categories = [le_caste.classes_[i] for i in pred_labels]
dict2["Caste"] = pred_categories
print(dict2['Name'])
print(dict2['Caste'])

Map:   0%|          | 0/666 [00:00<?, ? examples/s]

['ABHAY', 'PANCHAL MANOJ ', 'GONSALVES SUNNY', 'AMRIT', 'BAISOYA RISHABH', 'GIRI HEMANT', 'KADIAN YOGESH', 'SINGH HARBHEJ', 'SINGH JOGA', 'SHARMA SACHIN', 'BAJAJ RITIK', 'SINGH BARJINDER', 'KUMAR ANIL', 'PATEL PRITESH MUKESHBHAI', 'JAIN HARSHIT BABULAL', 'SINGH VIKRAMJEET', 'SANGWAN KAPIL', 'SINGH SATINDERJEET', 'SANDHU HARWINDER SINGH', 'SINGH RANDEEP', 'KOLANJA MOHAMMAD RAFEEQ', 'PURUSHAN SUDHEESH KUMAR', 'HAIDER SYED MUHAMMAD ARSHIYAN', 'NANAVATI ADITYA', 'SINGH GURJINDER', 'GUPTA VIKASH', 'SANDHU SHAMINDER SINGH', 'RAIJADA KAVALJITSINH MAHENDRASINH', 'SINGH HARSAHIB', 'ANGOM SANATOMBA SINGH', 'PUKHRAMBAM SAMANANDA SINGH', 'KSHETRIMAYUM BROJEN SINGH', 'TAKHELAMBAM RAGHU', 'THOKCHOM GANDHI SINGH', 'SULTAN AYAZ MOHAMMED', 'ARMAR MOHAMMAD SHAFI', 'SINGH KARANVIR', 'SINGH AVTAR', 'SINGH HARMEET', 'RANPARIYA JAYSUKH', 'SINGH JANG BAHADUR', 'SINGH GURMIT', 'BHANSALI MIHIR RASHMI', 'SINGH GURPREET', 'SINGH MEHENGA', 'LAMKANG STARSON', 'MOIRANGTHEM TAMBA', 'KHURAIJAM MAIPAK', 'MUNSHI SHAIKH

In [16]:
print(cluster1)

['sc/st', 'unknown', 'unknown', 'unknown', 'obc', 'unknown', 'unknown', 'muslim', 'muslim', 'muslim', 'sc/st', 'obc', 'obc', 'obc', 'muslim', 'obc', 'obc', 'sc/st', 'sc/st', 'general', 'obc', 'obc', 'muslim', 'general', 'obc', 'general', 'obc', 'sc/st', 'muslim', 'obc', 'unknown', 'obc', 'muslim', 'general', 'sc/st', 'muslim', 'obc', 'muslim', 'sc/st', 'unknown', 'sc/st', 'unknown', 'sc/st', 'sc/st', 'obc', 'general', 'unknown', 'general', 'sc/st', 'obc', 'general', 'obc', 'muslim', 'obc', 'muslim', 'obc', 'muslim', 'general', 'unknown', 'unknown', 'obc', 'general', 'sc/st', 'muslim', 'muslim', 'general', 'obc', 'unknown', 'unknown', 'muslim', 'sc/st', 'general', 'obc', 'general', 'obc', 'sc/st', 'sc/st', 'general', 'sc/st', 'general', 'obc', 'sc/st', 'obc', 'obc', 'general', 'obc', 'obc', 'general', 'sc/st', 'sc/st', 'unknown', 'sc/st', 'unknown', 'obc', 'obc', 'unknown', 'obc', 'obc', 'obc', 'obc', 'muslim', 'unknown', 'unknown', 'unknown', 'obc', 'muslim', 'sc/st', 'muslim', 'obc', 

In [17]:
dict3 = {'Name':name1,'Cluster_caste':cluster1}
dict2 = {'Name':name2}
train_dataset = Dataset.from_dict(dict3)
test_dataset = Dataset.from_dict(dict2)
dataset1 = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

from sklearn.preprocessing import LabelEncoder
le_cluster = LabelEncoder()
dataset1["train"] = dataset1["train"].add_column(
    "label_caste", le_cluster.fit_transform(dataset1["train"]["Cluster_caste"])
).remove_columns("Cluster_caste")
print(dataset1)
num_labels1 = len(le_cluster.classes_)
print(num_labels1)

DatasetDict({
    train: Dataset({
        features: ['Name', 'label_caste'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['Name'],
        num_rows: 666
    })
})
5


In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
def tokenize(examples):
    return tokenizer(
        examples["Name"],
        padding="max_length",
        truncation=True
    )

dataset1 = dataset1.map(tokenize, batched=True)
model = AutoModelForSequenceClassification.from_pretrained(
    "ai4bharat/indic-bert",
    num_labels=num_labels1
)
import numpy as np

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/666 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
predictions.decoder.bias         | UNEXPECTED | 
sop_classifier.classifier.weight | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
classifier.weight                | MISSING    | 
classifier.bias                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Name', 'label_caste', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['Name', 'input_ids', 'attention_mask'],
        num_rows: 666
    })
})


In [20]:
import evaluate
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)
from transformers import TrainingArguments, DataCollatorWithPadding # Re-add DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="yelp_review_classifiersed",
    eval_strategy="epoch",
    push_to_hub=False,
    warmup_steps=0.13,
    learning_rate = 5e-5,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=3e-8,
    weight_decay=0.001,
    lr_scheduler_type = "linear",
    num_train_epochs=50
)
from transformers import Trainer

split = dataset1["train"].train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]
print(train_dataset)
train_dataset = train_dataset.rename_column("label_caste", "labels")
eval_dataset = eval_dataset.rename_column("label_caste", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

trainer.train()



Dataset({
    features: ['Name', 'label_caste', 'input_ids', 'attention_mask'],
    num_rows: 4500
})


Epoch,Training Loss,Validation Loss,Accuracy
1,1.591388,1.348788,0.372000
2,1.295231,1.288689,0.414000
3,1.210892,1.190617,0.480000
4,1.160429,1.186389,0.498000
5,1.136459,1.180765,0.500000
6,1.106274,1.203605,0.446000
7,1.085818,1.202190,0.478000
8,1.017634,1.271417,0.492000
9,0.928835,1.237266,0.500000
10,0.855765,1.377575,0.474000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'Name': ['ABHAY', 'PANCHAL MANOJ ', 'GONSALVES SUNNY', 'AMRIT', 'BAISOYA RISHABH', 'GIRI HEMANT', 'KADIAN YOGESH', 'SINGH HARBHEJ', 'SINGH JOGA', 'SHARMA SACHIN', 'BAJAJ RITIK', 'SINGH BARJINDER', 'KUMAR ANIL', 'PATEL PRITESH MUKESHBHAI', 'JAIN HARSHIT BABULAL', 'SINGH VIKRAMJEET', 'SANGWAN KAPIL', 'SINGH SATINDERJEET', 'SANDHU HARWINDER SINGH', 'SINGH RANDEEP', 'KOLANJA MOHAMMAD RAFEEQ', 'PURUSHAN SUDHEESH KUMAR', 'HAIDER SYED MUHAMMAD ARSHIYAN', 'NANAVATI ADITYA', 'SINGH GURJINDER', 'GUPTA VIKASH', 'SANDHU SHAMINDER SINGH', 'RAIJADA KAVALJITSINH MAHENDRASINH', 'SINGH HARSAHIB', 'ANGOM SANATOMBA SINGH', 'PUKHRAMBAM SAMANANDA SINGH', 'KSHETRIMAYUM BROJEN SINGH', 'TAKHELAMBAM RAGHU', 'THOKCHOM GANDHI SINGH', 'SULTAN AYAZ MOHAMMED', 'ARMAR MOHAMMAD SHAFI', 'SINGH KARANVIR', 'SINGH AVTAR', 'SINGH HARMEET', 'RANPARIYA JAYSUKH', 'SINGH JANG BAHADUR', 'SINGH GURMIT', 'BHANSALI MIHIR RASHMI', 'SINGH GURPREET', 'SINGH MEHENGA', 'LAMKANG STARSON', 'MOIRANGTHEM TAMBA', 'KHURAIJAM MAIPAK', 'MUNS

In [22]:
pred_results = trainer.predict(dataset["test"])
pred_labels = np.argmax(pred_results.predictions, axis=-1)
pred_categories = [le_cluster.classes_[i] for i in pred_labels]
dict2["Cluster"] = pred_categories
print(dict2['Name'])
print(dict2['Cluster'])

['ABHAY', 'PANCHAL MANOJ ', 'GONSALVES SUNNY', 'AMRIT', 'BAISOYA RISHABH', 'GIRI HEMANT', 'KADIAN YOGESH', 'SINGH HARBHEJ', 'SINGH JOGA', 'SHARMA SACHIN', 'BAJAJ RITIK', 'SINGH BARJINDER', 'KUMAR ANIL', 'PATEL PRITESH MUKESHBHAI', 'JAIN HARSHIT BABULAL', 'SINGH VIKRAMJEET', 'SANGWAN KAPIL', 'SINGH SATINDERJEET', 'SANDHU HARWINDER SINGH', 'SINGH RANDEEP', 'KOLANJA MOHAMMAD RAFEEQ', 'PURUSHAN SUDHEESH KUMAR', 'HAIDER SYED MUHAMMAD ARSHIYAN', 'NANAVATI ADITYA', 'SINGH GURJINDER', 'GUPTA VIKASH', 'SANDHU SHAMINDER SINGH', 'RAIJADA KAVALJITSINH MAHENDRASINH', 'SINGH HARSAHIB', 'ANGOM SANATOMBA SINGH', 'PUKHRAMBAM SAMANANDA SINGH', 'KSHETRIMAYUM BROJEN SINGH', 'TAKHELAMBAM RAGHU', 'THOKCHOM GANDHI SINGH', 'SULTAN AYAZ MOHAMMED', 'ARMAR MOHAMMAD SHAFI', 'SINGH KARANVIR', 'SINGH AVTAR', 'SINGH HARMEET', 'RANPARIYA JAYSUKH', 'SINGH JANG BAHADUR', 'SINGH GURMIT', 'BHANSALI MIHIR RASHMI', 'SINGH GURPREET', 'SINGH MEHENGA', 'LAMKANG STARSON', 'MOIRANGTHEM TAMBA', 'KHURAIJAM MAIPAK', 'MUNSHI SHAIKH